# Generación de Datos — Bootcamp BI Analyst
## Programa de Afiliados Mercado Libre

Ejecutar las celdas **en orden**.  
Los únicos parámetros que hay que tocar son `N_AFFILIATES` y `N_JIRA_TOTAL` en la **Celda 1**.

| Tabla | Descripción |
|---|---|
| `LK_AFFILIATES` | Lookup de afiliados activos en el programa |
| `BT_REGISTERED_SOCIAL_MEDIA` | Redes sociales por afiliado |
| `BT_JIRA_HUNTING_AFILIADOS` | Tablero operativo del proceso de hunting |
| `BT_MARKETPLACE_AFFILIATE_URLS` | Links de afiliado generados por producto |
| `BT_MARKETPLACE_ACCESS_LOGS` | Clicks / accesos a links de afiliado |
| `BT_MARKETPLACE_SALES` | Ventas generadas por links de afiliado |

In [1]:
# ================================================================
# CELDA 1 — Imports & Configuración
# ================================================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

SEED = 42
rng  = np.random.default_rng(SEED)
random.seed(SEED)

OUTPUT_DIR = 'output_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TODAY      = datetime(2026, 3, 20)
JIRA_START = TODAY - timedelta(days=90)

PROGRAM_LAUNCH = {
    'BRA': TODAY - timedelta(days=365 * 3),
    'MEX': TODAY - timedelta(days=365 * 3),
    'ARG': TODAY - timedelta(days=365),
    'CHI': TODAY - timedelta(days=365),
}

# ── ESCALA (ajustar libremente) ────────────────────────────────
N_AFFILIATES = 500      # total de afiliados en el programa
N_JIRA_TOTAL = 1_000    # total de leads en el tablero Jira
# ──────────────────────────────────────────────────────────────

COUNTRIES = ['BRA', 'MEX', 'ARG', 'CHI']
COUNTRY_W = [0.30,  0.30,  0.20,  0.20]

CATEGORIES = [
    'lifestyle', 'home_deco', 'fitness', 'travel',
    'education', 'tech', 'beauty', 'food', 'other',
]

PLATFORMS = ['instagram', 'tiktok', 'youtube', 'facebook', 'x', 'other']

MELI_CODE = {'ARG': 'MLA', 'BRA': 'MLB', 'MEX': 'MLM', 'CHI': 'MLC'}

HUNTER_NAMES = [
    'Ana González', 'Carlos Mendez', 'Valentina Torres',
    'Bruno Alves',  'Sofía Ramírez',
]
_ht       = np.array([0.70, 0.85, 1.00, 1.25, 1.40])
HUNTER_W  = _ht / _ht.sum()               # pesos para asignación de leads
HUNTER_CR = [0.14, 0.17, 0.20, 0.23, 0.26]  # conversion rate por hunter

# Funnel Jira — proporciones exactas
JIRA_FUNNEL = {
    'pool':        0.60,
    'asignado':    0.10,
    'contactado':  0.10,
    'rechazado':   0.16,
    'afiliado':    0.04,
}

print('Configuración cargada.')
print('  Afiliados objetivo :', N_AFFILIATES)
print('  Leads Jira total   :', N_JIRA_TOTAL)
print('  Jira operativo     :', JIRA_START.date(), '->', TODAY.date())

Configuración cargada.
  Afiliados objetivo : 500
  Leads Jira total   : 1000
  Jira operativo     : 2025-12-20 -> 2026-03-20


In [2]:
# ================================================================
# CELDA 2 — Funciones auxiliares
# ================================================================

FIRST_NAMES = {
    'ARG': ['Valentina','Sofía','Martina','Agustina','Lucía',
            'Santiago','Mateo','Nicolás','Facundo','Tomás','Emma','Julián','Lautaro'],
    'CHI': ['Camila','Javiera','Constanza','Valentina','Isidora',
            'Diego','Sebastián','Matías','Ignacio','Felipe','Catalina','Bastián'],
    'BRA': ['Ana','Julia','Larissa','Fernanda','Beatriz',
            'Lucas','Gabriel','Matheus','Guilherme','Rafael','Isabela','João','Pedro'],
    'MEX': ['Fernanda','Valentina','Mariana','Daniela','Karla',
            'Diego','Miguel','Jorge','Andrés','Javier','Paola','Rodrigo'],
}
LAST_NAMES = {
    'ARG': ['García','Rodríguez','González','Fernández','López',
            'Martínez','Pérez','Sánchez','Romero','Torres','Díaz'],
    'CHI': ['González','Muñoz','Rojas','Díaz','Pérez',
            'Soto','Contreras','Silva','Martínez','Flores','Álvarez'],
    'BRA': ['Silva','Santos','Oliveira','Souza','Rodrigues',
            'Ferreira','Alves','Pereira','Lima','Gomes','Ribeiro'],
    'MEX': ['García','Martínez','Hernández','López','González',
            'Pérez','Sánchez','Ramírez','Torres','Flores','Morales'],
}

_ACCENT = str.maketrans({
    'á':'a','é':'e','í':'i','ó':'o','ú':'u',
    'à':'a','è':'e','ì':'i','ò':'o','ù':'u',
    'ä':'a','ë':'e','ï':'i','ö':'o','ü':'u',
    'ñ':'n','ç':'c',
    'Á':'A','É':'E','Í':'I','Ó':'O','Ú':'U',
    'Ä':'A','Ë':'E','Ï':'I','Ö':'O','Ü':'U','Ñ':'N',
})

def clean(s):
    return s.translate(_ACCENT).lower().replace(' ', '').replace('.', '')

def random_name(country):
    first = random.choice(FIRST_NAMES[country])
    last  = random.choice(LAST_NAMES[country])
    return first, last, first + ' ' + last

def make_username(first, last, salt):
    suffix = int(rng.integers(10, 9_999))
    return clean(first) + clean(last) + str(suffix)

def make_handle(first, last):
    r = int(rng.integers(0, 4))
    if r == 0: return clean(first) + '.' + clean(last)
    if r == 1: return clean(first) + '_' + clean(last)
    if r == 2: return clean(first) + str(int(rng.integers(10, 999)))
    return 'the.' + clean(first)

def follower_count(n=1):
    """Lognormal: mediana ≈ 36K, 95pct ≈ 260K. Rango [8K, 2M]."""
    raw = rng.lognormal(mean=10.5, sigma=1.2, size=n)
    return np.clip(raw, 8_000, 2_000_000).astype(int)

def rand_date(start, end):
    d = max(int((end - start).days), 1)
    return start + timedelta(days=int(rng.integers(0, d)))

def make_product_id(country):
    return MELI_CODE[country] + str(int(rng.integers(10_000_000, 99_999_999)))

def make_affiliate_url(username, country):
    pid = make_product_id(country)
    return 'https://meli.me/' + MELI_CODE[country].lower() + '/' + pid.lower() + '?aff=' + username, pid

print('Helpers listos.')

Helpers listos.


In [3]:
# ================================================================
# CELDA 3 — Leads Jira (todos los influencers identificados)
# ================================================================

n = N_JIRA_TOTAL
shuf = np.arange(n)
rng.shuffle(shuf)

# Conteos exactos por estado
n_pool = int(n * JIRA_FUNNEL['pool'])
n_asig = int(n * JIRA_FUNNEL['asignado'])
n_cont = int(n * JIRA_FUNNEL['contactado'])
n_rech = int(n * JIRA_FUNNEL['rechazado'])
n_afil = n - n_pool - n_asig - n_cont - n_rech

states = np.empty(n, dtype=object)
c0 = n_pool
c1 = c0 + n_asig
c2 = c1 + n_cont
c3 = c2 + n_rech
states[shuf[:c0]]  = 'pool'
states[shuf[c0:c1]] = 'asignado'
states[shuf[c1:c2]] = 'contactado'
states[shuf[c2:c3]] = 'rechazado'
states[shuf[c3:]]   = 'afiliado'

jira_rows = []
for i in range(n):
    country         = str(rng.choice(COUNTRIES, p=COUNTRY_W))
    first, last, nombre = random_name(country)
    state           = str(states[i])
    category        = random.choice(CATEGORIES)

    has_ig = random.random() < 0.85
    has_tt = random.random() < 0.72
    if not has_ig and not has_tt:
        has_ig = True
    ig_handle = make_handle(first, last) if has_ig else None
    tt_handle = make_handle(first, last) if has_tt else None

    hunter_idx      = None
    ultimo_contacto = None

    if state in ('asignado', 'contactado', 'afiliado', 'rechazado'):
        hunter_idx = int(rng.choice(len(HUNTER_NAMES), p=HUNTER_W))
        if state == 'asignado':
            # Asignado reciente, primera fecha de contacto/asignación
            ultimo_contacto = rand_date(JIRA_START, TODAY - timedelta(days=5))
        elif state == 'contactado':
            # Primer contacto hecho, pendiente segundo
            ultimo_contacto = rand_date(JIRA_START + timedelta(days=7), TODAY - timedelta(days=2))
        else:
            # afiliado / rechazado: segundo contacto = resolución final
            ultimo_contacto = rand_date(JIRA_START + timedelta(days=14), TODAY - timedelta(days=1))

    jira_rows.append({
        '_i':              i,
        '_state':          state,
        '_country':        country,
        '_first':          first,
        '_last':           last,
        '_category':       category,
        '_hunter_idx':     hunter_idx,
        'NOMBRE':          nombre,
        'INSTAGRAM':       ig_handle,
        'TIKTOK':          tt_handle,
        'ULTIMO_CONTACTO': ultimo_contacto.strftime('%Y-%m-%d') if ultimo_contacto else None,
    })

df_jira_raw = pd.DataFrame(jira_rows)

print('Leads Jira generados:', len(df_jira_raw))
print(df_jira_raw['_state'].value_counts().to_string())

Leads Jira generados: 1000
_state
pool          600
rechazado     160
contactado    100
asignado      100
afiliado       40


In [4]:
# ================================================================
# CELDA 4 — LK_AFFILIATES
# ================================================================

aff_records   = []
aff_id_seq    = [1]      # contador mutable
jira_aff_map  = {}       # _i → {AFFILIATE_ID, MELI_USERNAME}

# ── A) Afiliados provenientes de Jira (estado = afiliado) ──────
jira_aff_df = df_jira_raw[df_jira_raw['_state'] == 'afiliado'].reset_index(drop=True)

for _, row in jira_aff_df.iterrows():
    country  = row['_country']
    first    = row['_first']
    last     = row['_last']
    raw_i    = int(row['_i'])
    username = make_username(first, last, raw_i)
    uid      = int(rng.integers(10_000_000, 99_999_999))
    joined   = rand_date(JIRA_START, TODAY - timedelta(days=3))
    aid      = aff_id_seq[0]
    aff_id_seq[0] += 1

    jira_aff_map[raw_i] = {'AFFILIATE_ID': aid, 'MELI_USERNAME': username}

    aff_records.append({
        'AFFILIATE_ID':  aid,
        'MELI_USERNAME': username,
        'MELI_USER_ID':  uid,
        'COUNTRY':       country,
        'CATEGORY':      row['_category'],
        '_joined':       joined,
        '_first':        first,
        '_last':         last,
        '_source':       'jira',
    })

print('Afiliados Jira-sourced :', len(aff_records))

# ── B) Afiliados orgánicos ─────────────────────────────────────
n_organic = N_AFFILIATES - len(aff_records)

for i in range(n_organic):
    country        = str(rng.choice(COUNTRIES, p=COUNTRY_W))
    first, last, _ = random_name(country)
    username       = make_username(first, last, 100_000 + i)
    uid            = int(rng.integers(10_000_000, 99_999_999))
    joined         = rand_date(PROGRAM_LAUNCH[country], TODAY - timedelta(days=30))
    aid            = aff_id_seq[0]
    aff_id_seq[0] += 1

    aff_records.append({
        'AFFILIATE_ID':  aid,
        'MELI_USERNAME': username,
        'MELI_USER_ID':  uid,
        'COUNTRY':       country,
        'CATEGORY':      random.choice(CATEGORIES),
        '_joined':       joined,
        '_first':        first,
        '_last':         last,
        '_source':       'organic',
    })

df_affiliates = pd.DataFrame(aff_records)

# Quitar duplicados de MELI_USERNAME (muy improbable pero seguro)
df_affiliates = df_affiliates.drop_duplicates('MELI_USERNAME').reset_index(drop=True)
df_affiliates['AFFILIATE_ID'] = range(1, len(df_affiliates) + 1)

df_lk_affiliates = df_affiliates[
    ['AFFILIATE_ID', 'MELI_USERNAME', 'MELI_USER_ID', 'COUNTRY', 'CATEGORY']
].copy()

print('Total afiliados        :', len(df_lk_affiliates))
print('\nDistribución país:')
print(df_lk_affiliates['COUNTRY'].value_counts().to_string())
print('\nDistribución categoría:')
print(df_lk_affiliates['CATEGORY'].value_counts().to_string())

Afiliados Jira-sourced : 40
Total afiliados        : 500

Distribución país:
COUNTRY
BRA    157
MEX    145
ARG     99
CHI     99

Distribución categoría:
CATEGORY
fitness      64
education    64
other        59
food         58
lifestyle    54
tech         52
beauty       51
travel       51
home_deco    47


In [5]:
# ================================================================
# CELDA 5 — BT_REGISTERED_SOCIAL_MEDIA
# ================================================================

PLAT_PROB = {
    'instagram': 0.85,
    'tiktok':    0.72,
    'youtube':   0.32,
    'facebook':  0.28,
    'x':         0.22,
    'other':     0.08,
}
PLAT_URL_TPL = {
    'instagram': 'https://instagram.com/',
    'tiktok':    'https://tiktok.com/@',
    'youtube':   'https://youtube.com/@',
    'facebook':  'https://facebook.com/',
    'x':         'https://x.com/',
    'other':     'https://linktr.ee/',
}

sm_records = []
sm_id_seq  = [1]

for _, aff in df_affiliates.iterrows():
    first = aff['_first']
    last  = aff['_last']
    aid   = aff['AFFILIATE_ID']

    chosen = [p for p, prob in PLAT_PROB.items() if random.random() < prob]
    if not chosen:
        chosen = ['instagram']

    main_followers = int(follower_count(1)[0])

    for j, plat in enumerate(chosen):
        if j == 0:
            followers = main_followers
        else:
            ratio     = float(rng.uniform(0.08, 0.75))
            followers = max(1_000, int(main_followers * ratio))

        handle = make_handle(first, last)
        url    = PLAT_URL_TPL[plat] + handle

        sm_records.append({
            'SM_ID':        sm_id_seq[0],
            'AFFILIATE_ID': aid,
            'SOCIAL_MEDIA': plat,
            'URL':          url,
            'FOLLOWERS':    followers,
        })
        sm_id_seq[0] += 1

df_sm = pd.DataFrame(sm_records)

print('Registros social media :', len(df_sm))
print('Plataformas:')
print(df_sm['SOCIAL_MEDIA'].value_counts().to_string())
print('\nPromedio plataformas/afiliado :', round(len(df_sm) / len(df_affiliates), 1))
print('Distribución seguidores (plataforma principal):')
main_sm = df_sm.groupby('AFFILIATE_ID').first()['FOLLOWERS']
print(pd.cut(main_sm, bins=[0,10_000,50_000,100_000,500_000,2_100_000],
             labels=['8K-10K','10K-50K','50K-100K','100K-500K','500K-2M']).value_counts().sort_index().to_string())

Registros social media : 1226
Plataformas:
SOCIAL_MEDIA
instagram    429
tiktok       357
youtube      157
facebook     138
x            102
other         43

Promedio plataformas/afiliado : 2.5
Distribución seguidores (plataforma principal):
FOLLOWERS
8K-10K        73
10K-50K      232
50K-100K     102
100K-500K     86
500K-2M        7


In [ ]:
# ================================================================
# CELDA 6 — BT_JIRA_HUNTING_AFILIADOS
# ================================================================

# Reconstruir map actualizado de affiliate_id tras posible dedup
# Los MELI_USERNAMEs de jira_aff_map no cambiaron, solo el AFFILIATE_ID podría haber
# cambiado si hubo duplicados (muy improbable). Actualizamos por si acaso.
uname_to_aid = df_affiliates.set_index('MELI_USERNAME')['AFFILIATE_ID'].to_dict()
for raw_i, info in jira_aff_map.items():
    uname = info['MELI_USERNAME']
    if uname in uname_to_aid:
        jira_aff_map[raw_i]['AFFILIATE_ID'] = uname_to_aid[uname]

jira_out = []
for _, row in df_jira_raw.iterrows():
    state  = str(row['_state'])
    raw_i  = int(row['_i'])

    is_asig = state in ('asignado', 'contactado', 'afiliado', 'rechazado')
    is_cont = state in ('contactado', 'afiliado', 'rechazado')
    is_afil = state == 'afiliado'
    is_rech = state == 'rechazado'

    h_idx           = row['_hunter_idx']
    # pd.notna porque pandas convierte None a NaN (float) al guardar en DataFrame
    nombre_asignado = HUNTER_NAMES[int(h_idx)] if pd.notna(h_idx) else None

    meli_username = None
    if is_afil and raw_i in jira_aff_map:
        meli_username = jira_aff_map[raw_i]['MELI_USERNAME']

    jira_key = 'HUNT-' + str(raw_i + 1).zfill(4)

    jira_out.append({
        'JIRA_KEY':         jira_key,
        'MELI_USERNAME':    meli_username,
        'NOMBRE_ASIGNADO':  nombre_asignado,
        'NOMBRE':           row['NOMBRE'],
        'ULTIMO_CONTACTO':  row['ULTIMO_CONTACTO'],
        'INSTAGRAM':        row['INSTAGRAM'],
        'TIKTOK':           row['TIKTOK'],
        'asignado':         is_asig,
        'contactado':       is_cont,
        'afiliado':         is_afil,
        'rechazado':        is_rech,
    })

df_jira = pd.DataFrame(jira_out)

n_pool_out = (~df_jira['asignado']).sum()
n_asig_out = (df_jira['asignado'] & ~df_jira['contactado']).sum()
n_cont_out = (df_jira['contactado'] & ~df_jira['afiliado'] & ~df_jira['rechazado']).sum()
n_afil_out = df_jira['afiliado'].sum()
n_rech_out = df_jira['rechazado'].sum()

print('Registros Jira :', len(df_jira))
print('  pool          :', n_pool_out, '(' + str(round(100*n_pool_out/len(df_jira),1)) + '%)')
print('  asignado      :', n_asig_out, '(' + str(round(100*n_asig_out/len(df_jira),1)) + '%)')
print('  contactado    :', n_cont_out, '(' + str(round(100*n_cont_out/len(df_jira),1)) + '%)')
print('  afiliado      :', n_afil_out, '(' + str(round(100*n_afil_out/len(df_jira),1)) + '%)')
print('  rechazado     :', n_rech_out, '(' + str(round(100*n_rech_out/len(df_jira),1)) + '%)')
print('\nDistribución por hunter (leads asignados):')
print(df_jira[df_jira['asignado']]['NOMBRE_ASIGNADO'].value_counts().to_string())

In [ ]:
# ================================================================
# CELDA 7 — BT_MARKETPLACE_AFFILIATE_URLS
# ================================================================

url_records = []

for _, aff in df_affiliates.iterrows():
    aid      = aff['AFFILIATE_ID']
    username = aff['MELI_USERNAME']
    country  = aff['COUNTRY']
    joined   = aff['_joined']
    days_aff = max((TODAY - joined).days, 1)

    # Más URLs cuanto más tiempo llevan afiliados
    # log-normal con media proporcional a los meses activos
    mu_urls = np.log(max(days_aff / 25.0, 2.0))
    n_urls  = int(np.clip(rng.lognormal(mean=mu_urls, sigma=0.55), 2, 30))

    for _ in range(n_urls):
        url, pid = make_affiliate_url(username, country)
        created_at = rand_date(joined, TODAY - timedelta(days=1))
        is_closed  = random.random() < 0.28
        closed_at  = rand_date(created_at + timedelta(days=1), TODAY) if is_closed else None
        end_dt     = closed_at if closed_at else TODAY

        url_records.append({
            'URL':                    url,
            'AFFILIATE_ID':           aid,
            'MARKETPLACE_PRODUCT_ID': pid,
            'CREATED_AT':             created_at.strftime('%Y-%m-%d'),
            'CLOSED_AT':              closed_at.strftime('%Y-%m-%d') if closed_at else None,
            '_created_dt':            created_at,
            '_end_dt':                end_dt,
        })

df_urls = pd.DataFrame(url_records).drop_duplicates('URL').reset_index(drop=True)

print('URLs de afiliado       :', len(df_urls))
print('  Prom. por afiliado   :', round(len(df_urls) / len(df_affiliates), 1))
closed_n = df_urls['CLOSED_AT'].notna().sum()
print('  Cerradas             :', closed_n, '(' + str(round(100*closed_n/len(df_urls))) + '%)')

In [ ]:
# ================================================================
# CELDA 8 — BT_MARKETPLACE_ACCESS_LOGS  (vectorizado)
# ================================================================

# Máximo de seguidores por afiliado → calibra volumen de clicks
aff_max_followers = df_sm.groupby('AFFILIATE_ID')['FOLLOWERS'].max().to_dict()

# Convertir columnas de fecha a datetime
df_urls['_created_dt'] = pd.to_datetime(df_urls['_created_dt'])
df_urls['_end_dt']     = pd.to_datetime(df_urls['_end_dt'])
df_urls['_active_days'] = (
    (df_urls['_end_dt'] - df_urls['_created_dt']).dt.days.clip(lower=1)
)

# Max followers por URL
df_urls['_max_followers'] = df_urls['AFFILIATE_ID'].map(aff_max_followers).fillna(20_000)

# Clicks esperados:
# ~0.1% de seguidores por mes, distribuido entre todas las URLs del afiliado (~8 avg)
active_months = df_urls['_active_days'] / 30.0
mean_clicks   = (df_urls['_max_followers'] * 0.001 / 8.0) * active_months
log_mean      = np.log(np.maximum(mean_clicks.values, 1.5))
n_clicks_raw  = rng.lognormal(mean=log_mean, sigma=0.9)
df_urls['_n_clicks'] = np.clip(n_clicks_raw, 0, 5_000).astype(int)

total_clicks = int(df_urls['_n_clicks'].sum())
print('Total clicks a generar :', f'{total_clicks:,}')

# Expandir: una fila por click
active_mask = df_urls['_n_clicks'] > 0
df_u_active = df_urls[active_mask].copy()
rep_idx     = np.repeat(df_u_active.index.values, df_u_active['_n_clicks'].values)
df_acc_base = df_u_active.loc[rep_idx, ['URL', '_created_dt', '_end_dt']].reset_index(drop=True)

# Timestamps aleatorios dentro del período activo de cada URL
active_secs    = (
    (df_acc_base['_end_dt'] - df_acc_base['_created_dt']).dt.total_seconds().values
)
random_offsets = (rng.random(len(df_acc_base)) * active_secs).astype('int64')
df_acc_base['ACCESS_DATETIME'] = (
    df_acc_base['_created_dt'] + pd.to_timedelta(random_offsets, unit='s')
)

# Pool de buyers (usuarios que hacen click en los links)
n_buyers   = max(total_clicks // 5, 10_000)
buyer_ids  = rng.integers(1_000_000, 9_999_999, size=n_buyers)
buyer_pool = np.array(['buyer_' + str(x) for x in buyer_ids])
b_idx      = rng.integers(0, n_buyers, size=len(df_acc_base))
df_acc_base['MELI_USERNAME'] = buyer_pool[b_idx]

df_access_logs = df_acc_base[['URL', 'MELI_USERNAME', 'ACCESS_DATETIME']].copy()
df_access_logs['ACCESS_DATETIME'] = (
    df_access_logs['ACCESS_DATETIME'].dt.strftime('%Y-%m-%d %H:%M:%S')
)

print('Registros access_logs  :', f'{len(df_access_logs):,}')

In [ ]:
# ================================================================
# CELDA 9 — BT_MARKETPLACE_SALES  (vectorizado)
# ================================================================

CONVERSION_RATE = 0.030    # 3% de clicks se convierten en venta

# Rangos de precio en moneda local
PRICE_RANGES = {
    'ARG': (3_000,   300_000),   # ARS
    'BRA': (   30,     3_000),   # BRL
    'MEX': (  150,    15_000),   # MXN
    'CHI': (2_000,   200_000),   # CLP
}

# Mapear URL → país a través de AFFILIATE_ID
url_to_country = (
    df_urls[['URL', 'AFFILIATE_ID']]
    .merge(df_affiliates[['AFFILIATE_ID', 'COUNTRY']], on='AFFILIATE_ID')
    .set_index('URL')['COUNTRY']
    .to_dict()
)

# Muestrear conversiones
sale_mask     = rng.random(len(df_access_logs)) < CONVERSION_RATE
df_sales_base = df_access_logs[sale_mask].copy().reset_index(drop=True)

# Datetime de compra = acceso + 1 a 45 minutos
df_sales_base['ACCESS_DATETIME']  = pd.to_datetime(df_sales_base['ACCESS_DATETIME'])
min_offset = rng.integers(1, 45, size=len(df_sales_base))
df_sales_base['PURCHASE_DATETIME'] = (
    df_sales_base['ACCESS_DATETIME'] + pd.to_timedelta(min_offset, unit='m')
).clip(upper=pd.Timestamp(TODAY))

# País y precio
df_sales_base['COUNTRY'] = df_sales_base['URL'].map(url_to_country).fillna('BRA')
prices = np.zeros(len(df_sales_base))
for country, (lo, hi) in PRICE_RANGES.items():
    mask = (df_sales_base['COUNTRY'] == country).values
    if mask.sum() > 0:
        prices[mask] = np.round(rng.uniform(lo, hi, size=int(mask.sum())), 2)
df_sales_base['PRICE'] = prices

df_sales = df_sales_base[['URL', 'MELI_USERNAME', 'PURCHASE_DATETIME', 'PRICE']].copy()
df_sales['PURCHASE_DATETIME'] = df_sales['PURCHASE_DATETIME'].dt.strftime('%Y-%m-%d %H:%M:%S')

real_cr = round(100 * len(df_sales) / len(df_access_logs), 1)
print('Registros ventas       :', f'{len(df_sales):,}')
print('Conv. rate efectivo    :', str(real_cr) + '%')
print('\nPrecio promedio por país:')
print(df_sales_base.groupby('COUNTRY')['PRICE'].agg(['mean','min','max']).round(0).to_string())

In [ ]:
# ================================================================
# CELDA 10 — Exportar CSVs
# ================================================================

EXPORT_COLS = {
    'LK_AFFILIATES': [
        'AFFILIATE_ID', 'MELI_USERNAME', 'MELI_USER_ID', 'COUNTRY', 'CATEGORY'
    ],
    'BT_REGISTERED_SOCIAL_MEDIA': [
        'SM_ID', 'AFFILIATE_ID', 'SOCIAL_MEDIA', 'URL', 'FOLLOWERS'
    ],
    'BT_JIRA_HUNTING_AFILIADOS': [
        'JIRA_KEY', 'MELI_USERNAME', 'NOMBRE_ASIGNADO', 'NOMBRE',
        'ULTIMO_CONTACTO', 'INSTAGRAM', 'TIKTOK',
        'asignado', 'contactado', 'afiliado', 'rechazado'
    ],
    'BT_MARKETPLACE_AFFILIATE_URLS': [
        'URL', 'AFFILIATE_ID', 'MARKETPLACE_PRODUCT_ID', 'CREATED_AT', 'CLOSED_AT'
    ],
    'BT_MARKETPLACE_ACCESS_LOGS': [
        'URL', 'MELI_USERNAME', 'ACCESS_DATETIME'
    ],
    'BT_MARKETPLACE_SALES': [
        'URL', 'MELI_USERNAME', 'PURCHASE_DATETIME', 'PRICE'
    ],
}

DATAFRAMES = {
    'LK_AFFILIATES':                 df_lk_affiliates,
    'BT_REGISTERED_SOCIAL_MEDIA':    df_sm,
    'BT_JIRA_HUNTING_AFILIADOS':     df_jira,
    'BT_MARKETPLACE_AFFILIATE_URLS': df_urls,
    'BT_MARKETPLACE_ACCESS_LOGS':    df_access_logs,
    'BT_MARKETPLACE_SALES':          df_sales,
}

print('Exportando CSVs a:', OUTPUT_DIR)
print('-' * 65)
total_rows = 0
for name, df in DATAFRAMES.items():
    cols = EXPORT_COLS[name]
    out  = df[cols].copy()
    path = OUTPUT_DIR + '/' + name + '.csv'
    out.to_csv(path, index=False)
    rows = len(out)
    total_rows += rows
    print(f'  {name:<42} {rows:>8,} filas   →  {path}')

print('-' * 65)
print('Total filas generadas  :', f'{total_rows:,}')
print('\nTodo listo.')